# Notebook 03 — Segmentación SamGeo sobre AOI acotado (versión vigente)

Aplica el modelo SamGeo con backbone vit_b sobre los composites RGB recortados al AOI acotado (SFF CGSM + Vía Parque Isla de Salamanca, 835 km²) y reproyecta los GeoJSON resultantes al sistema oficial colombiano EPSG:9377.

Esta es la **versión limpia** del flujo de segmentación. El notebook `03_segmentation.ipynb` original contiene el flujo legacy sobre AOI envolvente (5.073 km²) y se conserva para trazabilidad metodológica del baseline (Anexo C del informe).

**Insumos:**
- `data/processed/rgb_acotado/CGSM_RGB_{degradacion,recuperacion,actual}.tif`

**Productos:**
- `data/processed/samgeo_acotado/mask_*.tif`
- `data/processed/samgeo_acotado/manglar_*.geojson` (EPSG:4326)
- `data/processed/samgeo_acotado/manglar_*_9377.geojson` (EPSG:9377)

**Patrones del curso aplicados:**
- Tarea 18: lectura perezosa de los GeoTIFF de máscara
- Tarea 19: reproyección al sistema oficial colombiano MAGNA-SIRGAS Origen Nacional

In [ ]:
import sys, os
sys.path.insert(0, '../src')
from pathlib import Path
import rasterio
from rasterio.enums import Resampling
from samgeo import SamGeo
from python.utils import raster_metadata, vector_to_9377, EPSG_NACIONAL

ROOT = Path('..').resolve()
RGB_DIR = ROOT / 'data' / 'processed' / 'rgb_acotado'
RES_DIR = RGB_DIR / 'resampled'
OUT_DIR = ROOT / 'data' / 'processed' / 'samgeo_acotado'
RES_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

PERIODOS = ['degradacion', 'recuperacion', 'actual']
print(f'Entrada: {RGB_DIR}')
print(f'Salida:  {OUT_DIR}')

## 1. Verificación lazy de los TIF de RGB (Tarea 18)

Lectura perezosa para confirmar dimensiones, CRS y resolución sin materializar la grilla en memoria.

In [ ]:
for nombre in PERIODOS:
    p = RGB_DIR / f'CGSM_RGB_{nombre}.tif'
    if not p.exists():
        print(f'  no encontrado: {p}'); continue
    meta = raster_metadata(p)
    print(f'  {nombre:13s} → shape={meta["shape"]}, res={meta["res"]}, crs={meta["crs"]}')

## 2. Resampling de 10 m a 30 m para SamGeo

El backbone vit_b de SamGeo requiere reducir resolución para no agotar la RAM del contenedor (~8 GB).

In [ ]:
for nombre in PERIODOS:
    src_path = RGB_DIR / f'CGSM_RGB_{nombre}.tif'
    dst_path = RES_DIR / f'CGSM_RGB_{nombre}_30m.tif'
    if dst_path.exists():
        print(f'  ya existe: {dst_path.name}'); continue
    with rasterio.open(src_path) as src:
        scale = 10 / 30
        new_w, new_h = int(src.width * scale), int(src.height * scale)
        data = src.read(out_shape=(src.count, new_h, new_w),
                        resampling=Resampling.bilinear)
        transform = src.transform * src.transform.scale(
            src.width / new_w, src.height / new_h)
        meta = src.meta.copy()
        meta.update({'height': new_h, 'width': new_w, 'transform': transform})
        with rasterio.open(dst_path, 'w', **meta) as dst:
            dst.write(data)
    print(f'  resampleado: {dst_path.name}')

## 3. Segmentación con SamGeo vit_b

Modelo de fundación SAM (Meta AI, Kirillov et al. 2023) adaptado a datos geoespaciales (Wu y Osco 2023). Tiempo esperado: 5-15 min por período.

In [ ]:
sam = SamGeo(model_type='vit_b', automatic=True)

for nombre in PERIODOS:
    rgb = RES_DIR / f'CGSM_RGB_{nombre}_30m.tif'
    mask_out = OUT_DIR / f'mask_{nombre}.tif'
    geojson_out = OUT_DIR / f'manglar_{nombre}.geojson'
    if mask_out.exists() and geojson_out.exists():
        print(f'  ya procesado: {nombre}'); continue
    print(f'\nSegmentando {nombre}...')
    sam.generate(str(rgb), output=str(mask_out))
    sam.tiff_to_vector(str(mask_out), str(geojson_out))
    print(f'  máscara: {mask_out.name}')
    print(f'  vector:  {geojson_out.name}')

print('\n*** SamGeo terminado ***')

## 4. Reproyección a EPSG:9377 (Tarea 19)

Los GeoJSON salen de SamGeo en EPSG:4326. Se reproyectan al sistema oficial colombiano MAGNA-SIRGAS Origen Nacional para que el cálculo de áreas en hectáreas se realice sobre una proyección equivalente para Colombia.

In [ ]:
for nombre in PERIODOS:
    src = OUT_DIR / f'manglar_{nombre}.geojson'
    dst = OUT_DIR / f'manglar_{nombre}_9377.geojson'
    if not src.exists():
        print(f'  no encontrado: {src}'); continue
    vector_to_9377(src, dst)
    print(f'  {nombre}: reproyectado a {EPSG_NACIONAL} → {dst.name}')

## 5. Áreas en hectáreas y resumen

Cálculo del área de cada parche directamente en metros sobre EPSG:9377, sin aproximaciones esféricas.

In [ ]:
import geopandas as gpd
import pandas as pd
from python.utils import area_ha_9377

filas = []
for nombre in PERIODOS:
    p = OUT_DIR / f'manglar_{nombre}_9377.geojson'
    gdf = area_ha_9377(gpd.read_file(p))
    gdf_f = gdf[(gdf['area_ha'] >= 1.0) & (gdf['area_ha'] < 5000.0)]
    filas.append({
        'periodo': nombre,
        'parches_totales': int(len(gdf)),
        'parches_filtrados': int(len(gdf_f)),
        'area_ha_filtrada': round(float(gdf_f['area_ha'].sum()), 1),
        'area_media_ha': round(float(gdf_f['area_ha'].mean()), 2),
    })

df = pd.DataFrame(filas)
df.to_csv(ROOT / 'outputs' / 'tables' / 'areas_9377_acotado.csv', index=False)
print(df.to_string(index=False))
print('\nGuardado: outputs/tables/areas_9377_acotado.csv')